 # In class assignment #2: Modeling ecological communities with joint species distribution models (JDSMs)

#### As stated on the very first day of class, one of the goals of this course is for students to understand not only the basic principles of interactions + coevolutionary theory, but to understand *how we study* interactions using a variety of methods. Many of you have asked me about how often I use coding in my research. My answer: ***All the time.***

#### In an effort to expose you to coding & evolutionary and ecological modeling, I am including two "in-class assignments" in the course, both of which will be an interactive coding notebook. This is the second notebook, which focuses on species distribution models and joint species distribution models. *You do not need to have any coding experience to complete this assignment*. I will walk you through how to use the notebook, and ask you to detail your observations and think through those observations in the context of what we have learned in class.

<span style="color:darkred">

**Assignment instructions:**

This assignment is worth **50 points**. There are 15 questions worth 2 points each and 5 "reflection questions" at the end of the assignment worth 4 points each**. This assignment is different from in-class assignment 1 because we are working with ***real data***. For this reason, I will not have you toggle any numbers. Instead, I walk you through the datasets, show you how to fit an SDM and JDSM, then ask you to contextualize your findings in light of a real-world example of disease dynamics.

Remember:
Each block of code is contained within a **cell** which are run individually. Press shift + enter to run each cell. Most of the code cells need to be run in sequence, so you will get an error if you try to run one before running the previous one. Because of confusion related to package loading last time, I have included the necessary packages at the top of *every code cell*, that way, you won't need to remember to reload packages every time!

There are a couple of cells that throw warnings (there's a warning message in red), but still run just fine. Don't freak out if you see that, it's totally fine! Just ignore it. If your cells throw errors and don't complete their run, then you have an problem.

You may work with a partner on this assignment, but I expect each of you to record your answers independently. So I know who worked together, please ***record your partner's name at the end***.

As I said last time **MAKE SURE TO SAVE YOUR WORK AS YOU GO ALONG**. Copy-paste answers into a separate document just in case your progress gets erased. Binder is great for sharing code, but this is one of its major limitations. A way to overcome this is to download the .ipynb file you've been working on and then re-upload it to your binder space when you want to continue. If you don't know what this means or how to make it happen, just keep track of things in a separate document. I have instructions for downloading the notebook as an html file at the end for Courseworks submission, but I'm also fine with you submitting a word doc or pdf with your responses and screenshots as relevant!

</span>

<div class="alert alert-success">
<b>Learning objectives</b>
<br />1. Understand the conceptual difference between a species distribution model (SDM) and joint species distribution model (JDSM)
<br />2. Understand the typical outputs of a JDSM
<br />3. Evaluate how species interactions impact community ecology
<br />4. Evaluate the utility of SDMs/JDSMs with respect to real-world problems like disease, food security, and biodiversity loss.
<div>

### Modeling the Distribution of White-Tailed Deer

Imagine you are an ecologist interested in understanding the distribution of white-tailed deer (*Odocoileus virginianus*) across North America.

White-tailed deer populations have surged in many areas, particularly in suburban and urban environments, leading to ecological, economic, and safety concerns.

To understand and predict their distribution, you and your collaborators have collected:

---

#### 1. Occurrence data
- Species observations

#### 2. Environmental data
We will focus on two key climate variables:

- **Mean Annual Temperature (°C)**
- **Annual Precipitation (mm)**

These variables are commonly used in species distribution models because they strongly influence:
- Plant productivity (food availability)
- Winter survival
- Habitat suitability

---
We will combine these datasets to build a species distribution model (SDM).



**If you're curious:** To get data to work with here, I queried two databases: GBIF (Global Biodiversity Information Facility) for occurrence data and WorldClim for climate data. I used a random sub-sample of deer occurrence records so that this code would run fast and be easy to work with. I pre-assembled a dataframe that is hosted in the same Github repository as the jupyter notebook so we can easily load it in and work with it!

## Part 0: Grab the data

**First, import your occurrence + climate dataset for *O. virginianus***. I have randomly sampled 5000 *O. virginanus* occurrence records from the northeastern United States and their corresponding climate data. Here, we're importing and cleaning that dataframe.

In [ ]:
# =========================
# LOAD + CLEAN ALL DATA
# =========================

import pandas as pd
import numpy as np

# --- Load datasets ---
df = pd.read_csv("deer_sdm_NE.csv")
bg = pd.read_csv("background_sdm_NE.csv")
grid_points = pd.read_csv("grid_sdm_NE.csv")

# --- Define cleaning function ---
def clean_climate(data):
    data = data.copy()
    
    # remove infinities
    data = data.replace([np.inf, -np.inf], np.nan)
    
    # remove clearly invalid climate values
    data.loc[data['precip'] < 0, 'precip'] = np.nan
    data.loc[(data['temp'] < -50) | (data['temp'] > 50), 'temp'] = np.nan
    
    return data

# --- Apply cleaning ---
df = clean_climate(df)
bg = clean_climate(bg)
grid_points = clean_climate(grid_points)

# --- Drop missing values ---
df = df.dropna(subset=['temp','precip'])
bg = bg.dropna(subset=['temp','precip'])
grid_points = grid_points.dropna(subset=['temp','precip'])

# =========================
# DISPLAY DEER DATA
# =========================

df

**Question 1: Describe the dataframe: what are its dimensions? What data is included? Why do we need this data to run an SDM?** (remember, to answer the questions double-click into the "Answer" cells and begin typing)

Answer:

#### Now I can plot my occurrence data on a map of the eastern U.S.

In [ ]:
import geopandas as gpd
import geodatasets
import matplotlib.pyplot as plt

world = gpd.read_file(geodatasets.get_path("naturalearth.land"))

gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df.lon, df.lat),
    crs="EPSG:4326"
)

fig, ax = plt.subplots(figsize=(8,6))

world.plot(ax=ax, color='lightgray')
gdf.plot(ax=ax, color='black', markersize=3)

plt.xlim(-80, -66)
plt.ylim(38, 48)
plt.title("White-tailed deer occurrences (Northeast US)")

plt.show()

**Question 2: Describe the distribution of *O. virginianus*. Is this consistent with your own knowledge about where *O. virginianus* can be found?**

Answer:

## Part 1: Basic SDMs

Below, I build a basic SDM including only deer occurrence and climate data

In [ ]:
#encode presence/absence

df['presence'] = 1
bg['presence'] = 0

full = pd.concat([df, bg]).dropna()

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt

# =========================
# LOAD DATA (already cleaned!)
# =========================

df = pd.read_csv("deer_sdm_NE.csv")
bg = pd.read_csv("background_sdm_NE.csv")
grid_points = pd.read_csv("grid_sdm_NE.csv")

# =========================
# ADD PRESENCE
# =========================

df['presence'] = 1
bg['presence'] = 0

# combine
full = pd.concat([df, bg])

#clean
full = full.replace([np.inf, -np.inf], np.nan)
full = full.dropna(subset=['temp','precip'])

# =========================
# FIT MODEL
# =========================

X = full[['temp','precip']]
y = full['presence']

model = LogisticRegression(max_iter=1000)
model.fit(X, y)

# =========================
# CLEAN GRID
# =========================

grid_points_clean = grid_points.replace([np.inf, -np.inf], np.nan)
grid_points_clean = grid_points_clean.dropna(subset=['temp','precip'])

# =========================
# PREDICT
# =========================

pred = model.predict_proba(
    grid_points_clean[['temp','precip']]
)[:,1]

# =========================
# REBUILD FULL MAP
# =========================

pred_full = np.full(len(grid_points), np.nan)
pred_full[grid_points_clean.index] = pred

n_lon = len(np.unique(grid_points['lon']))
n_lat = len(np.unique(grid_points['lat']))

pred_map = pred_full.reshape((n_lat, n_lon))

# =========================
# PLOT
# =========================

plt.figure(figsize=(8,6))

plt.imshow(
    pred_map,
    extent=[-80, -66, 38, 48],
    origin='lower'
)

plt.colorbar(label="Habitat suitability")

plt.scatter(df['lon'], df['lat'], s=3, color='black')

plt.title("Species Distribution Model (SDM): White-tailed deer")
plt.xlabel("Longitude")
plt.ylabel("Latitude")

plt.show()

**Question 3: What do the colors on the plot mean?**

Answer:

**Question 4: Which part of this plot is the actual model output? What information was used to get that output?**

Answer:

**Question 5: What do you notice about the points with respect to habitat suitability?**

Answer:

**Question 6: How well do you think this SDM is working? What are some ways we could improve its function?**

Answer:

## Part 2: Adding in biotic interactions with JDSMs

So far, we have modeled the distribution of white-tailed deer (*Odocoileus virginianus*) using only environmental variables (temperature and precipitation). This approach assumes that climate alone determines where deer can occur.

However, species do not exist in isolation—they interact with other species in their environment.

In this section, we will incorporate a key ecological interaction:

### Predator–prey interaction
The coyote (*Canis latrans*) is a widespread predator in North America. Coyotes can influence deer populations by preying on fawns and occasionally adults, reducing survival and recruitment. As a result, predator presence can shape where deer populations persist across the landscape.

### From SDM to JDSM
Previously, we built a **Species Distribution Model (SDM)**:
> Deer distribution ~ climate (temperature + precipitation)

Now, we will extend this to include a biotic interaction:
> Deer distribution ~ climate + predator influence

We will approximate predator influence using **distance to the nearest coyote observation**. This allows us to incorporate information about another species into our model.

This type of model is often referred to as a **Joint Species Distribution Model (JDSM)**, because it accounts for multiple species simultaneously.

### Key idea
By including coyotes, we move from predicting where deer *could* live based on climate, to where they are more likely to *actually occur* given both environmental conditions and biotic interactions.

---

**Question 7: How do you think adding in coyote occurrence data will change the predicted distribution of *O. virginianus*? Why?**

Answer:

**Question 8: When I add in the coyote data, I don't just add coyote presence, instead I calculate something I called "coyote_dist". Take a look at the code below to check out where I add "coyote_dist". Describe what "coyote_dist" is and why we generate this variable?**

Answer:

In [ ]:
## Fitting the JDSM

from scipy.spatial import cKDTree
from sklearn.linear_model import LogisticRegression
import numpy as np
import pandas as pd

# --- Load and format coyote data ---
coyote = pd.read_csv("coyote_presence_NE.csv")

coyote = coyote.rename(columns={
    'decimalLongitude': 'lon',
    'decimalLatitude': 'lat'
})

# --- Build KDTree ---
tree = cKDTree(coyote[['lon','lat']].values)

# --- Add coyote distance ---
df['coyote_dist'] = tree.query(df[['lon','lat']].values, k=1)[0]
bg['coyote_dist'] = tree.query(bg[['lon','lat']].values, k=1)[0]

# --- Clean datasets ---
df = clean_climate(df)
bg = clean_climate(bg)

# --- Log-transform distance ---
df['coyote_dist'] = np.log1p(df['coyote_dist'])
bg['coyote_dist'] = np.log1p(bg['coyote_dist'])

# --- Add presence labels ---
df['presence'] = 1
bg['presence'] = 0

# --- Balance dataset (with replacement if needed) ---
bg = bg.sample(len(df), replace=True, random_state=42)

# --- Combine ---
full = pd.concat([df, bg]).dropna()

# --- Fit JDSM ---
X = full[['temp','precip','coyote_dist']]
y = full['presence']

jdsm = LogisticRegression(max_iter=1000)
jdsm.fit(X, y)

# --- Add coyote distance to grid ---
grid_points['coyote_dist'] = tree.query(grid_points[['lon','lat']].values, k=1)[0]

# --- Clean grid ---
grid_points = clean_climate(grid_points)

# --- Log-transform grid ---
grid_points['coyote_dist'] = np.log1p(grid_points['coyote_dist'])

grid_points_clean = grid_points.dropna(subset=['temp','precip','coyote_dist'])

# --- Predict ---
pred_jdsm = jdsm.predict_proba(
    grid_points_clean[['temp','precip','coyote_dist']]
)[:,1]

# --- Rebuild map ---
pred_full = np.full(len(grid_points), np.nan)
pred_full[grid_points_clean.index] = pred_jdsm

n_lon = len(np.unique(grid_points['lon']))
n_lat = len(np.unique(grid_points['lat']))

pred_map_jdsm = pred_full.reshape((n_lat, n_lon))

In [ ]:
#Mapping the JDSM results

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14,6))

# --- SDM ---
im1 = axes[0].imshow(
    pred_map,
    extent=[-80, -66, 38, 48],
    origin='lower',
    alpha=0.9
)

axes[0].scatter(
    df['lon'], df['lat'],
    s=10,
    color='black',
    edgecolor='black',
    linewidth=0.4
)

axes[0].set_title("SDM (Climate only)")
axes[0].set_xlabel("Longitude")
axes[0].set_ylabel("Latitude")

# --- JDSM ---
im2 = axes[1].imshow(
    pred_map_jdsm,
    extent=[-80, -66, 38, 48],
    origin='lower',
    alpha=0.9
)

axes[1].scatter(
    df['lon'], df['lat'],
    s=10,
    color='black',
    edgecolor='black',
    linewidth=0.4
)

axes[1].set_title("JDSM (Climate + Coyotes)")
axes[1].set_xlabel("Longitude")

# --- Create space for colorbar ---
fig.subplots_adjust(right=0.88)

# --- Add colorbar OUTSIDE ---
cbar_ax = fig.add_axes([0.90, 0.15, 0.02, 0.7])  # [left, bottom, width, height]
cbar = fig.colorbar(im2, cax=cbar_ax)
cbar.set_label("Habitat suitability")

plt.suptitle("Effect of Biotic Interactions on Species Distributions", y=1.02)

plt.show()

**Question 9: Compare the two maps. Where are areas of high suitability in each model? Are the overall patterns similar or different? (Remember, which species' distribution are we modeling here? What do the colors correspond to?)**

Answer:

**Question 10: Which model appears smoother? Why might that be?**

Answer:

**Question 11: Which model do you think is a better fit for the data? Why?**

Answer:

**Question 12: What do you predict coyote distribution looks like? Describe it based on the deer habitat suitability in the JDSM.**

Answer:

***Check to see if your prediction is accurate below:***

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

fig = plt.figure(figsize=(18,10))

# --- Create grid with space for colorbar ---
gs = gridspec.GridSpec(
    2, 3,
    height_ratios=[1, 1],
    width_ratios=[1, 1, 1],
    hspace=0.3,
    wspace=0.25
)

extent = [-80, -66, 38, 48]

# =========================
# TOP ROW
# =========================

ax1 = fig.add_subplot(gs[0,0])
ax2 = fig.add_subplot(gs[0,1])
ax3 = fig.add_subplot(gs[0,2])

# Deer
ax1.scatter(df['lon'], df['lat'], s=6, color='black', alpha=0.6)
ax1.set_title("White-tailed deer\n(Odocoileus virginianus)")
ax1.set_xlim(extent[0], extent[1])
ax1.set_ylim(extent[2], extent[3])
ax1.set_xlabel("Longitude")
ax1.set_ylabel("Latitude")

# Coyote
ax2.scatter(coyote['lon'], coyote['lat'], s=6, color='gray', alpha=0.6)
ax2.set_title("Coyote\n(Canis latrans)")
ax2.set_xlim(extent[0], extent[1])
ax2.set_ylim(extent[2], extent[3])
ax2.set_xlabel("Longitude")

# Overlay
ax3.scatter(df['lon'], df['lat'], s=6, color='black', alpha=0.5, label='Deer')
ax3.scatter(coyote['lon'], coyote['lat'], s=6, color='gray', alpha=0.5, label='Coyote')
ax3.set_title("Deer + Coyote overlap")
ax3.set_xlim(extent[0], extent[1])
ax3.set_ylim(extent[2], extent[3])
ax3.set_xlabel("Longitude")
ax3.legend()

# =========================
# BOTTOM CENTER: JDSM
# =========================

ax4 = fig.add_subplot(gs[1,1])

im = ax4.imshow(
    pred_map_jdsm,
    extent=extent,
    origin='lower'
)

ax4.scatter(df['lon'], df['lat'],
            s=5, color='white', edgecolor='black', linewidth=0.3, label='Deer')

ax4.scatter(coyote['lon'], coyote['lat'],
            s=5, color='black', alpha=0.7, label='Coyote')

ax4.set_title("JDSM with species occurrences")
ax4.set_xlim(extent[0], extent[1])
ax4.set_ylim(extent[2], extent[3])
ax4.set_xlabel("Longitude")
ax4.set_ylabel("Latitude")
ax4.legend()

# turn off empty panels
fig.add_subplot(gs[1,0]).axis('off')
fig.add_subplot(gs[1,2]).axis('off')

# =========================
# COLORBAR (far right, clean)
# =========================

cbar_ax = fig.add_axes([0.92, 0.25, 0.015, 0.5])  # moved further right
cbar = fig.colorbar(im, cax=cbar_ax)
cbar.set_label("Habitat suitability")

# =========================
# TITLE (bigger + no overlap)
# =========================

fig.suptitle(
    "Species Occurrence and Modeled Distribution",
    fontsize=18,
    y=0.98   # push higher so it doesn't overlap
)

plt.show()

**Question 13: Was your prediction correct? What do you notice about the relationship between coyotes and deer? Note: unlike the Lotka-Volterra models, this model does not explicitly state that coyotes should have a negative impact on deer population. Given this, are you surprised by what you see?**

Answer:

## Part 3: Linking Species Distributions to Disease Risk

So far, we have explored how environmental conditions and species interactions (e.g., predation by coyotes) shape the distribution of white-tailed deer (*Odocoileus virginianus*). 

In this section, we extend this idea to a broader ecological and societal context: **infectious disease dynamics**.

---

### The Lyme disease system

Lyme disease is a vector-borne disease caused by the bacterium *Borrelia burgdorferi*. It is transmitted to humans through the bite of infected blacklegged ticks (*Ixodes scapularis*) in eastern North America.

This system involves multiple interacting species:

- **Ticks** (*Ixodes scapularis*) – the vector that transmits the pathogen  
- **Reservoir hosts** (e.g., small mammals like white-footed mice, *Peromyscus leucopus*) – maintain and amplify the pathogen  
- **Large mammals** (e.g., white-tailed deer, *Odocoileus virginianus*) – support adult tick populations  

---

### The role of white-tailed deer

White-tailed deer play a critical ecological role in the Lyme disease system:

- Adult ticks feed and reproduce on deer  
- Higher deer densities can support larger tick populations  
- More ticks can increase the likelihood of human exposure  

Importantly:
> Deer do **not** transmit the pathogen directly, but they strongly influence the abundance of ticks that do.

---

### Connecting ecology to disease

This means that:
- The **distribution of deer** can influence  
- The **distribution of ticks**, which affects  
- The **risk of Lyme disease in humans**

---

### Key idea

Disease risk is not determined by climate alone—it emerges from a network of interacting species.

In the next section, we will compare our model predictions of deer distributions to **real-world Lyme disease data** to explore how ecological patterns scale up to affect human health.

---

**Question 12: How do you predict deer distribution will be related to lyme disease occurrence? In your explanation, include the terms: *interaction network* and *indirect effects***

Answer:

#### Ok, lets check it out. First, we need to load in the lyme disease prevelance data from the CDC: 

In [ ]:
import geopandas as gpd
import numpy as np
import pandas as pd

lyme = pd.read_csv("Lyme disease public use aggregated data with geography, 2022-2023.csv")
lyme.head()

lyme = lyme[lyme['FIPS'] != 'Suppressed']
lyme['FIPS'] = lyme['FIPS'].astype(str).str.zfill(5)
lyme_county = lyme.groupby('FIPS')['Frequency'].sum().reset_index()


counties = gpd.read_file(
    "https://www2.census.gov/geo/tiger/GENZ2020/shp/cb_2020_us_county_500k.zip"
)

counties['FIPS'] = counties['STATEFP'] + counties['COUNTYFP']
lyme_map = counties.merge(lyme_county, on='FIPS', how='left')
lyme_map = lyme_map.cx[-80:-66, 38:48]

# ensure numeric
lyme_map['Frequency'] = pd.to_numeric(lyme_map['Frequency'], errors='coerce')

# create log-transformed column
lyme_map['cases_log'] = np.log1p(lyme_map['Frequency'])

#### Next, we can plot our JDSM and Lyme disease data side-by-side:

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import geopandas as gpd
import numpy as np
import pandas as pd

# --- Load state boundaries ---
states = gpd.read_file(
    "https://www2.census.gov/geo/tiger/GENZ2020/shp/cb_2020_us_state_500k.zip"
)

# subset to same extent
states = states.cx[-80:-66, 38:48]

# --- Ensure Lyme data is clean ---
lyme_map['Frequency'] = pd.to_numeric(lyme_map['Frequency'], errors='coerce')
lyme_map['cases_log'] = np.log1p(lyme_map['Frequency'])

# optional: drop missing
lyme_map = lyme_map.dropna(subset=['cases_log'])

# --- Create figure ---
fig, axes = plt.subplots(1, 2, figsize=(14,6))

extent = [-80, -66, 38, 48]

# =====================
# LEFT: JDSM
# =====================
im1 = axes[0].imshow(
    pred_map_jdsm,
    extent=extent,
    origin='lower',
    aspect='auto'   # ensures same scaling
)

# state boundaries
states.boundary.plot(
    ax=axes[0],
    color='white',
    linewidth=1
)

axes[0].set_xlim(extent[0], extent[1])
axes[0].set_ylim(extent[2], extent[3])

axes[0].set_title("Predicted deer distribution (JDSM)")
axes[0].set_xlabel("Longitude")
axes[0].set_ylabel("Latitude")

# =====================
# RIGHT: LYME
# =====================
lyme_map.plot(
    column='cases_log',
    cmap='Reds',
    linewidth=0.1,
    edgecolor='black',
    legend=False,
    ax=axes[1]
)

# state boundaries
states.boundary.plot(
    ax=axes[1],
    color='black',
    linewidth=1
)

axes[1].set_xlim(extent[0], extent[1])
axes[1].set_ylim(extent[2], extent[3])

axes[1].set_title("Lyme disease (log cases)")
axes[1].set_xlabel("Longitude")

# =====================
# COLORBAR (outside)
# =====================
fig.subplots_adjust(right=0.88)

norm = mpl.colors.Normalize(
    vmin=lyme_map['cases_log'].min(),
    vmax=lyme_map['cases_log'].max()
)

sm = mpl.cm.ScalarMappable(cmap='Reds', norm=norm)
sm._A = []

cbar_ax = fig.add_axes([0.90, 0.2, 0.02, 0.6])
cbar = fig.colorbar(sm, cax=cbar_ax)
cbar.set_label("Log Lyme cases")

# =====================
# FINAL TOUCH
# =====================
plt.suptitle("Ecological Predictions vs. Observed Disease Patterns", y=1.02)

plt.show()

**Question 14: Was your prediction correct, or are you surprised by what you see? In answering this question, point out specific geographic areas that either align with or deviate from your expectations.**

Answer:

**Question 15: There are a few counties that don't have lyme disease data in New York (one on Long Island and one upstate), Pennsylvania, and Vermont. Based on the *O. virginianus* JDSM, what do you predict the true lyme prevalence is in these counties? Call them NY1 (upstate), NY2 (long island), PA, and VT**

Answer:

## Reflections on JDSMs and this coding notebook

The following questions are worth 5 points each, and should be answered with a minimum of 3-4 sentences.

**Reflection Question 1: Are you convinced of the utility of JDSMs? Do you think biotic interactions add a necessary element to species distribution modeling, why or why not?**

Answer:

**Reflection Question 2: I gave you a real-world example showing the relevance of species distributions and species interaction networks to human health. Can you think of another example where SDMs or JDSMs can be linked to questions related to conservation, food security, or biodiversity loss? Explain one below. (Just describe the system you're thinking about)**

Answer:

**Reflection Question 3: If you were designing a small study or exercise like the one you just went through, but now looking at the system you described above, what would you do? What sorts of questions would you ask? What data would you pull in? What plots/maps would you want to make? Would you want to use an SDM or JDSM? Why?**

Answer:

**Reflection Question 4: Use this space to reflect on this assignment. Did you find it useful to get hands-on experience with species distribution models? What parts were hard or confusing? What questions do you stil have? What was it like to work with the code?**

Answer:

## Assignment Submission

If you worked with a partner, please record their name here.

**My partner:**

<div class="alert alert-success">
<b>Submitting the assignment</b>
<br /> When you are finished with a notebook and ready to submit it, you will download an HTML version of the notebook to submit. To do this, follow the instructions below. Upload the HTML notebook to the 'In-class Assignment 2' assignment in the In-class assignments part of the Courseworks to submit your assignment. These will be due Friday, April 3rd by 11:59pm.
<div>

Run the following cell and ignore any warnings:

In [ ]:
# =========================
# EXPORT NOTEBOOK TO HTML
# =========================

import subprocess

notebook_name = "JSDMs.ipynb"  # change if needed

cmd = [
    "jupyter", "nbconvert",
    notebook_name,
    "--to", "html",
    "--no-input",
    "--HTMLExporter.embed_images=True"
]

subprocess.run(cmd)

print("✅ HTML export complete! Check the file browser to download your .html file.")

Now download your file:

1. Click the file icon in the top left corner of your screen. This will reveal a list of files.
2. Find `JSDMs.html`
3. Right click it and then hit "Download"
4. A working html will download to your computer's "Downloads" folder.
5. Submit that file to the In-class Assignment 2 "Assignment" on Courseworks by Friday, April 3rd at 11:59pm